In [ ]:
import numpy as np
import pandas as pd
import json

np.random.seed(42)

# 1. LOAD & FEATURE ENGINEERING
df = pd.read_csv("Dengue_Climate_Bangladesh.csv")
df["DATE"]     = pd.to_datetime(df[["YEAR", "MONTH"]].assign(DAY=1))
df             = df.sort_values("DATE").reset_index(drop=True)
df["TIME_IDX"] = np.arange(len(df))

df["TEMP_RANGE"] = df["MAX"] - df["MIN"]
df["TEMP_MEAN"]  = (df["MAX"] + df["MIN"]) / 2
df["HEAT_INDEX"] = (df["TEMP_MEAN"] * df["HUMIDITY"]) / 100
df["RAIN_LOG"]   = np.log1p(df["RAINFALL"])
df["SIN_MONTH"]  = np.sin(2 * np.pi * df["MONTH"] / 12)
df["COS_MONTH"]  = np.cos(2 * np.pi * df["MONTH"] / 12)
df["DENGUE_LOG"] = np.log1p(df["DENGUE"])

# Lag features - each row only uses PAST information, no leakage
for lag in [1, 2, 3]:
    df[f"DENGUE_LAG{lag}"] = df["DENGUE_LOG"].shift(lag)
    df[f"RAIN_LAG{lag}"]   = df["RAIN_LOG"].shift(lag)
    df[f"HUMID_LAG{lag}"]  = df["HUMIDITY"].shift(lag)

df = df.dropna().reset_index(drop=True)

FEATURES = [
    "TEMP_MEAN", "TEMP_RANGE", "HEAT_INDEX",
    "HUMIDITY", "RAIN_LOG", "RAINFALL",
    "SIN_MONTH", "COS_MONTH", "TIME_IDX",
    "DENGUE_LAG1", "DENGUE_LAG2", "DENGUE_LAG3",
    "RAIN_LAG1", "RAIN_LAG2", "HUMID_LAG1",
]
TARGET     = "DENGUE_LOG"
N_FEATURES = len(FEATURES)
LOOKBACK   = 12

print("=" * 70)
print("  SHALLOW LSTM (NumPy) - DENGUE PREDICTION, BANGLADESH")
print("=" * 70)
print(f"Observations after feature engineering : {len(df)}")
print(f"Features                                : {N_FEATURES}")
print(f"Lookback window                         : {LOOKBACK} months")

# 2. TRAIN / TEST SPLIT  (<=2024 train, 2025 Jan-Oct test)
train_mask = df["YEAR"] <= 2024
test_mask  = df["YEAR"] == 2025

df_train = df[train_mask].copy()
df_test  = df[test_mask].copy()
print(f"Training months  : {len(df_train)}  (2008-2024)")
print(f"Test months      : {len(df_test)}  (2025 Jan-Oct)")

# 3. SCALING  (fit ONLY on training data -> no leakage)
class MinMaxScaler:
    def fit(self, X):
        self.min_ = X.min(axis=0)
        self.max_ = X.max(axis=0)
        self.range_ = np.where(self.max_ - self.min_ == 0, 1.0, self.max_ - self.min_)
        return self

    def transform(self, X):
        return (X - self.min_) / self.range_

    def fit_transform(self, X):
        return self.fit(X).transform(X)

    def inverse_transform(self, X):
        return X * self.range_ + self.min_


feat_scaler   = MinMaxScaler()
target_scaler = MinMaxScaler()

X_train_raw = df_train[FEATURES].values.astype(np.float64)
y_train_raw = df_train[TARGET].values.reshape(-1, 1).astype(np.float64)

X_train_sc = feat_scaler.fit_transform(X_train_raw)
y_train_sc = target_scaler.fit_transform(y_train_raw).ravel()

X_test_raw = df_test[FEATURES].values.astype(np.float64)
y_test_raw = df_test[TARGET].values.reshape(-1, 1).astype(np.float64)

# test transformed using TRAIN scalers only (correct, no leakage)
X_test_sc = feat_scaler.transform(X_test_raw)
y_test_sc = target_scaler.transform(y_test_raw).ravel()

# 4. SEQUENCE BUILDER
def build_sequences(X_sc, y_sc, lookback):
    Xs, ys = [], []
    for i in range(lookback, len(X_sc)):
        Xs.append(X_sc[i - lookback:i])
        ys.append(y_sc[i])
    return np.array(Xs), np.array(ys)

X_all_sc = np.vstack([X_train_sc, X_test_sc])
y_all_sc = np.concatenate([y_train_sc, y_test_sc])

X_seq, y_seq = build_sequences(X_all_sc, y_all_sc, LOOKBACK)

n_test  = len(df_test)
n_train = len(X_seq) - n_test

X_tr, y_tr = X_seq[:n_train], y_seq[:n_train]
X_te, y_te = X_seq[n_train:], y_seq[n_train:]

print(f"LSTM train sequences : X{X_tr.shape}  y{y_tr.shape}")
print(f"LSTM test  sequences : X{X_te.shape}  y{y_te.shape}")

# chronological validation split (last 15% of train, no shuffling -> time series safe)
val_frac = 0.15
n_val = max(1, int(len(X_tr) * val_frac))
X_fit, y_fit = X_tr[:-n_val], y_tr[:-n_val]
X_val, y_val = X_tr[-n_val:], y_tr[-n_val:]
print(f"Fit sequences: {len(X_fit)}   Val sequences: {len(X_val)} (chronological, no leakage)")

# 5. SHALLOW LSTM  (single recurrent layer) - pure NumPy, manual BPTT
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -60, 60)))

def huber_loss(y_true, y_pred, delta=1.0):
    err = y_true - y_pred
    abs_err = np.abs(err)
    quad = np.minimum(abs_err, delta)
    lin  = abs_err - quad
    return np.mean(0.5 * quad ** 2 + delta * lin)

def huber_grad(y_true, y_pred, delta=1.0):
    err = y_pred - y_true
    grad = np.where(np.abs(err) <= delta, err, delta * np.sign(err))
    return grad / len(y_true)


class ShallowLSTM:
    """
    Input (T, F) -> LSTM(H) -> Dropout -> Dense(32, relu) -> Dense(1, linear)
    """

    def __init__(self, n_features, hidden=64, dense_units=32,
                 dropout=0.2, lr=1e-3, l2_reg=1e-4, seed=42):
        rng = np.random.default_rng(seed)
        self.F, self.H, self.D = n_features, hidden, dense_units
        self.dropout = dropout
        self.lr = lr
        self.l2 = l2_reg

        z = n_features + hidden
        glorot = lambda fan_in, fan_out: rng.normal(0, np.sqrt(2.0 / (fan_in + fan_out)), (fan_in, fan_out))

        self.Wf, self.Wi, self.Wo, self.Wg = (glorot(z, hidden) for _ in range(4))
        self.bf = np.ones(hidden)      # forget gate bias init = 1 (helps training)
        self.bi = np.zeros(hidden)
        self.bo = np.zeros(hidden)
        self.bg = np.zeros(hidden)

        # Dense head
        self.W1 = glorot(hidden, dense_units)
        self.b1 = np.zeros(dense_units)
        self.W2 = glorot(dense_units, 1)
        self.b2 = np.zeros(1)

        self._params = ["Wf", "Wi", "Wo", "Wg", "bf", "bi", "bo", "bg",
                         "W1", "b1", "W2", "b2"]
        self._m = {p: np.zeros_like(getattr(self, p)) for p in self._params}
        self._v = {p: np.zeros_like(getattr(self, p)) for p in self._params}
        self._t = 0


    def forward(self, X, training=False):
        """X: (B, T, F). Returns predictions (B,) and cache for backward."""
        B, T, F = X.shape
        H = self.H
        h = np.zeros((B, H))
        c = np.zeros((B, H))
        cache = {"h": [h], "c": [c], "gates": [], "z": []}

        for t in range(T):
            x_t = X[:, t, :]
            zcat = np.concatenate([x_t, h], axis=1)          # (B, F+H)
            f = sigmoid(zcat @ self.Wf + self.bf)
            i = sigmoid(zcat @ self.Wi + self.bi)
            o = sigmoid(zcat @ self.Wo + self.bo)
            g = np.tanh(zcat @ self.Wg + self.bg)
            c = f * c + i * g
            h = o * np.tanh(c)
            cache["h"].append(h)
            cache["c"].append(c)
            cache["gates"].append((f, i, o, g))
            cache["z"].append(zcat)

        h_last = cache["h"][-1]

        # inverted dropout
        if training and self.dropout > 0:
            keep = 1.0 - self.dropout
            mask = (np.random.rand(*h_last.shape) < keep) / keep
            h_drop = h_last * mask
        else:
            mask = np.ones_like(h_last)
            h_drop = h_last

        d1_pre = h_drop @ self.W1 + self.b1
        d1 = np.maximum(d1_pre, 0)              # ReLU
        out = (d1 @ self.W2 + self.b2).ravel()  # linear output

        cache.update(dict(h_last=h_last, mask=mask, h_drop=h_drop,
                           d1_pre=d1_pre, d1=d1, X=X))
        return out, cache


    def backward(self, cache, grad_out):
        """grad_out: dLoss/dOut, shape (B,)."""
        B = grad_out.shape[0]
        grads = {p: np.zeros_like(getattr(self, p)) for p in self._params}

        grad_out = grad_out.reshape(-1, 1)             # (B,1)
        grads["W2"] += cache["d1"].T @ grad_out + self.l2 * self.W2
        grads["b2"] += grad_out.sum(axis=0)

        d_d1 = grad_out @ self.W2.T                     # (B, D)
        d_d1_pre = d_d1 * (cache["d1_pre"] > 0)          # ReLU'
        grads["W1"] += cache["h_drop"].T @ d_d1_pre + self.l2 * self.W1
        grads["b1"] += d_d1_pre.sum(axis=0)

        d_h_drop = d_d1_pre @ self.W1.T
        d_h_last = d_h_drop * cache["mask"]

        T = len(cache["gates"])
        dh_next = d_h_last
        dc_next = np.zeros((B, self.H))

        for t in reversed(range(T)):
            f, i, o, g = cache["gates"][t]
            c_t = cache["c"][t + 1]
            c_prev = cache["c"][t]
            h_prev = cache["h"][t]
            zcat = cache["z"][t]
            tanh_c = np.tanh(c_t)

            do = dh_next * tanh_c
            dc = dh_next * o * (1 - tanh_c ** 2) + dc_next

            df = dc * c_prev
            di = dc * g
            dg = dc * i
            dc_prev = dc * f

            df_raw = df * f * (1 - f)
            di_raw = di * i * (1 - i)
            do_raw = do * o * (1 - o)
            dg_raw = dg * (1 - g ** 2)

            grads["Wf"] += zcat.T @ df_raw
            grads["Wi"] += zcat.T @ di_raw
            grads["Wo"] += zcat.T @ do_raw
            grads["Wg"] += zcat.T @ dg_raw
            grads["bf"] += df_raw.sum(axis=0)
            grads["bi"] += di_raw.sum(axis=0)
            grads["bo"] += do_raw.sum(axis=0)
            grads["bg"] += dg_raw.sum(axis=0)

            dzcat = (df_raw @ self.Wf.T + di_raw @ self.Wi.T +
                     do_raw @ self.Wo.T + dg_raw @ self.Wg.T)
            dh_prev = dzcat[:, self.F:]

            dh_next = dh_prev
            dc_next = dc_prev

        for p in ["Wf", "Wi", "Wo", "Wg"]:
            grads[p] += self.l2 * getattr(self, p)

        # normalize by batch size
        for p in self._params:
            grads[p] /= B
        return grads


    def adam_step(self, grads, clipnorm=1.0):
        self._t += 1
        beta1, beta2, eps = 0.9, 0.999, 1e-8

        # global gradient clipping
        total_norm = np.sqrt(sum(np.sum(g ** 2) for g in grads.values()))
        if total_norm > clipnorm:
            scale = clipnorm / (total_norm + 1e-8)
            for p in grads:
                grads[p] *= scale

        for p in self._params:
            g = grads[p]
            self._m[p] = beta1 * self._m[p] + (1 - beta1) * g
            self._v[p] = beta2 * self._v[p] + (1 - beta2) * (g ** 2)
            m_hat = self._m[p] / (1 - beta1 ** self._t)
            v_hat = self._v[p] / (1 - beta2 ** self._t)
            update = self.lr * m_hat / (np.sqrt(v_hat) + eps)
            setattr(self, p, getattr(self, p) - update)


    def predict(self, X):
        out, _ = self.forward(X, training=False)
        return out

    def fit(self, X_fit, y_fit, X_val, y_val, epochs=300, batch_size=16,
            patience=25, verbose=True, tag=""):
        n = len(X_fit)
        best_val = np.inf
        best_state = None
        wait = 0
        history = {"loss": [], "val_loss": []}

        for epoch in range(1, epochs + 1):
            idx = np.arange(n)          # NOTE: no shuffle -> time series safe
            epoch_losses = []
            for start in range(0, n, batch_size):
                bidx = idx[start:start + batch_size]
                Xb, yb = X_fit[bidx], y_fit[bidx]
                pred, cache = self.forward(Xb, training=True)
                loss = huber_loss(yb, pred)
                grad = huber_grad(yb, pred)
                grads = self.backward(cache, grad)
                self.adam_step(grads)
                epoch_losses.append(loss)

            train_loss = float(np.mean(epoch_losses))
            val_pred = self.predict(X_val)
            val_loss = huber_loss(y_val, val_pred)
            history["loss"].append(train_loss)
            history["val_loss"].append(val_loss)

            if val_loss < best_val - 1e-6:
                best_val = val_loss
                best_state = {p: getattr(self, p).copy() for p in self._params}
                wait = 0
            else:
                wait += 1

            if verbose and (epoch % 20 == 0 or epoch == 1):
                print(f"    [{tag}] epoch {epoch:3d}  loss={train_loss:.5f}  val_loss={val_loss:.5f}")

            if wait >= patience:
                if verbose:
                    print(f"    [{tag}] early stopping at epoch {epoch} (best val_loss={best_val:.5f})")
                break

        if best_state is not None:
            for p in self._params:
                setattr(self, p, best_state[p])
        return history


# 6. LIGHTWEIGHT HYPERPARAMETER SEARCH (walk-forward, leakage-free)
def time_series_splits(n, n_splits=3, min_train=60):
    """Simple expanding-window walk-forward split, analogous to sklearn TimeSeriesSplit."""
    fold_size = (n - min_train) // n_splits
    splits = []
    for k in range(1, n_splits + 1):
        train_end = min_train + fold_size * k - fold_size
        val_end = train_end + fold_size
        if val_end > n:
            break
        splits.append((np.arange(0, train_end), np.arange(train_end, val_end)))
    return splits


print("\n" + "-" * 70)
print("  HYPERPARAMETER SEARCH (walk-forward CV, leakage-free)")
print("-" * 70)

param_grid = [
    {"hidden": 32, "dropout": 0.1, "lr": 1e-3, "l2_reg": 1e-4},
    {"hidden": 64, "dropout": 0.2, "lr": 1e-3, "l2_reg": 1e-4},
    {"hidden": 64, "dropout": 0.3, "lr": 5e-4, "l2_reg": 1e-4},
]

splits = time_series_splits(len(X_tr), n_splits=3, min_train=int(len(X_tr) * 0.5))
best_params, best_rmse = None, np.inf
search_rows = []

for params in param_grid:
    fold_rmses = []
    for k, (tr_idx, va_idx) in enumerate(splits):
        Xf_tr, yf_tr = X_tr[tr_idx], y_tr[tr_idx]
        Xf_va, yf_va = X_tr[va_idx], y_tr[va_idx]
        m = ShallowLSTM(N_FEATURES, **params)
        m.fit(Xf_tr, yf_tr, Xf_va, yf_va, epochs=120, batch_size=16,
              patience=15, verbose=False, tag=f"h{params['hidden']}-fold{k}")
        pred_sc = m.predict(Xf_va)
        pred_log = target_scaler.inverse_transform(pred_sc.reshape(-1, 1)).ravel()
        true_log = target_scaler.inverse_transform(yf_va.reshape(-1, 1)).ravel()
        rmse = np.sqrt(np.mean((true_log - pred_log) ** 2))
        fold_rmses.append(rmse)

    mean_rmse = float(np.mean(fold_rmses))
    search_rows.append({**params, "CV_RMSE_mean": mean_rmse})
    print(f"  hidden={params['hidden']:3d} dropout={params['dropout']:.1f} "
          f"lr={params['lr']:.0e} -> CV RMSE(log)={mean_rmse:.4f}")
    if mean_rmse < best_rmse:
        best_rmse = mean_rmse
        best_params = params

print(f"\nBest params: {best_params}  (CV RMSE(log)={best_rmse:.4f})")

# 7. FINAL MODEL - trained on full training set, evaluated on held-out 2025
print("\n" + "-" * 70)
print("  FINAL MODEL TRAINING (train: 2008-2024, forecast: 2025 Jan-Oct)")
print("-" * 70)

model = ShallowLSTM(N_FEATURES, **best_params)
history = model.fit(X_fit, y_fit, X_val, y_val, epochs=400, batch_size=16,
                     patience=35, verbose=True, tag="final")

print(f"\nTraining stopped at epoch {len(history['loss'])}")
print(f"Best val_loss (Huber): {min(history['val_loss']):.6f}")

# 8. PREDICTIONS  (inverse scale -> inverse log1p -> case counts)
def inv_transform_pred(pred_sc):
    log_vals = target_scaler.inverse_transform(pred_sc.reshape(-1, 1)).ravel()
    return np.expm1(log_vals)

lstm_train_pred = inv_transform_pred(model.predict(X_tr))
lstm_test_pred  = inv_transform_pred(model.predict(X_te))

y_train_orig = np.expm1(target_scaler.inverse_transform(y_tr.reshape(-1, 1)).ravel())
y_test_orig  = np.expm1(target_scaler.inverse_transform(y_te.reshape(-1, 1)).ravel())


# 9. METRICS
def pearson_r(a, b):
    a = a - a.mean(); b = b - b.mean()
    return float(np.sum(a * b) / (np.sqrt(np.sum(a ** 2)) * np.sqrt(np.sum(b ** 2)) + 1e-12))

def compute_metrics(y_true, y_pred, label):
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    mae  = float(np.mean(np.abs(y_true - y_pred)))
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    r2 = float(1 - ss_res / ss_tot)
    mape = float(np.mean(np.abs((y_true - y_pred) / (y_true + 1))) * 100)
    corr = pearson_r(y_true, y_pred)
    nse = r2  # identical formula to R^2 here
    return {"Model": label, "RMSE": rmse, "MAE": mae, "R2": r2,
            "NSE": nse, "MAPE_pct": mape, "Pearson_r": corr}

metrics_rows = [
    compute_metrics(y_train_orig, lstm_train_pred, "Shallow LSTM - Train (2008-2024)"),
    compute_metrics(y_test_orig, lstm_test_pred, "Shallow LSTM - Test (2025 Jan-Oct)"),
]
metrics_df = pd.DataFrame(metrics_rows)

print("\n" + "-" * 70)
print("  PERFORMANCE METRICS")
print("-" * 70)
print(metrics_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

metrics_df.to_csv("lstm_metrics.csv", index=False)
pd.DataFrame(search_rows).to_csv("lstm_hyperparameter_search.csv", index=False)

MONTH_NAMES = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

pred_2025 = df_test.copy()[["YEAR", "MONTH", "MIN", "MAX", "HUMIDITY", "RAINFALL", "DENGUE"]]
pred_2025.columns = ["Year", "Month", "Min_Temp", "Max_Temp", "Humidity", "Rainfall", "Actual_Cases"]
pred_2025["Month_Name"] = pred_2025["Month"].apply(lambda m: MONTH_NAMES[m - 1])
pred_2025["LSTM_Predicted"] = lstm_test_pred.astype(int)
pred_2025["Residual"] = pred_2025["Actual_Cases"] - pred_2025["LSTM_Predicted"]
pred_2025["AbsErr_Pct"] = (pred_2025["Residual"].abs() / (pred_2025["Actual_Cases"] + 1) * 100).round(1)
pred_2025.to_csv("lstm_2025_prediction_table.csv", index=False)

print("\n" + "-" * 70)
print("  2025 MONTHLY FORECAST")
print("-" * 70)
print(pred_2025[["Month_Name", "Actual_Cases", "LSTM_Predicted",
                  "Residual", "AbsErr_Pct"]].to_string(index=False))

# Save model weights + config for reproducibility
np.savez("best_lstm_model.npz",
         **{p: getattr(model, p) for p in model._params},
         hidden=model.H, features=N_FEATURES, lookback=LOOKBACK)

with open("model_config.json", "w") as f:
    json.dump({"best_params": best_params, "features": FEATURES,
                "lookback": LOOKBACK, "epochs_run": len(history["loss"])}, f, indent=2)

print("\n" + "=" * 70)
print("  DONE. Outputs saved:")
print("    lstm_metrics.csv")
print("    lstm_hyperparameter_search.csv")
print("    lstm_2025_prediction_table.csv")
print("    best_lstm_model.npz")
print("    model_config.json")
print("=" * 70)

  SHALLOW LSTM (NumPy) - DENGUE PREDICTION, BANGLADESH
Observations after feature engineering : 211
Features                                : 15
Lookback window                         : 12 months
Training months  : 201  (2008-2024)
Test months      : 10  (2025 Jan-Oct)
LSTM train sequences : X(189, 12, 15)  y(189,)
LSTM test  sequences : X(10, 12, 15)  y(10,)
Fit sequences: 161   Val sequences: 28 (chronological, no leakage)

----------------------------------------------------------------------
  HYPERPARAMETER SEARCH (walk-forward CV, leakage-free)
----------------------------------------------------------------------
  hidden= 32 dropout=0.1 lr=1e-03 -> CV RMSE(log)=1.5493
  hidden= 64 dropout=0.2 lr=1e-03 -> CV RMSE(log)=1.4415
  hidden= 64 dropout=0.3 lr=5e-04 -> CV RMSE(log)=1.5448

Best params: {'hidden': 64, 'dropout': 0.2, 'lr': 0.001, 'l2_reg': 0.0001}  (CV RMSE(log)=1.4415)

----------------------------------------------------------------------
  FINAL MODEL TRAINING (train